# Observed first-attempt work-mode trajectories

This notebook describes how first-attempt success and selected exercise difficulty evolve across `playlist` and `zpdes` module sequences. It follows the filtering logic of `work_mode_progress_random_module_model.ipynb`, works directly with the combined population, and reports observed rates rather than fitted model predictions.

The sequence unit is a **contiguous student × module × work-mode run**. Activity changes do not reset the sequence; attempt position restarts whenever the module or work mode changes. At each attempt number, the notebook displays the observed success rate and the mean calibrated Elo difficulty of the exercises presented.

## 1. Setup

The existing loader preserves the previous notebook's inclusion rules. The helpers construct the combined first-attempt trajectory and aggregate observed success and exercise Elo by work mode and attempt number.

In [37]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.model_work_mode_first_attempt_trajectory as trajectory_model  # noqa: E402

importlib.reload(trajectory_model)

from scripts.model_work_mode_first_attempt_trajectory import (  # noqa: E402
    build_empirical_trajectory_summary,
    load_mia_first_attempt_trajectory,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)

## 2. Parameters

There is no global minimum number of exercises per student. `MIN_SEQUENCE_EXERCISES = 31` retains uninterrupted student/module/work-mode sequences containing more than 30 unique first-attempt exercises. `MIN_SEQUENCES_PER_POSITION` and `MIN_STUDENTS_PER_POSITION` control only the displayed common range; the exported descriptive table retains every observed position.

In [38]:
INPUT_FILE = PROJECT_ROOT / 'data_MIA' / '986-neurips-mia_20260415_100024.parquet'
EXERCISE_CATALOG_JSON = PROJECT_ROOT / 'data_MIA' / 'exo_mia.json'
MODULE_CONFIG_JSON = PROJECT_ROOT / 'data_MIA' / 'config_mia.json'
EXERCISE_ELO_FILE = PROJECT_ROOT / 'artifacts' / 'sources' / 'mia' / 'artifacts' / 'derived' / 'agg_exercise_elo.parquet'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'first_attempt_trajectory_mia_notebook'

MIN_SEQUENCE_EXERCISES = 31
SEQUENCE_SCOPE = 'module'
KEEP_ONLY_SINGLE_MODULE_PLAYLISTS = False
MODEL_POPULATION = 'combined'
MIN_SEQUENCES_PER_POSITION = 100
MIN_STUDENTS_PER_POSITION = 100
SMOOTHING_WINDOW = 15

for required_path in (INPUT_FILE, EXERCISE_CATALOG_JSON, MODULE_CONFIG_JSON, EXERCISE_ELO_FILE):
    if not required_path.exists():
        raise FileNotFoundError(required_path)


## 3. Reproduce the existing filters for the combined population

A DuckDB query reproduces the previous notebook's raw-MIA preparation: keep `playlist` and `zpdes`, retain all playlist rows even when a playlist spans multiple modules, and map missing playlist hierarchy through UUID-backed activity `learning_items` in `config_mia.json`. There is no global student exercise threshold. All eligible students are loaded directly into one combined compact trajectory.

In [39]:
print('Preparing combined population...')
dataset = load_mia_first_attempt_trajectory(
    input_file=INPUT_FILE,
    exercise_catalog_json=EXERCISE_CATALOG_JSON,
    module_config_json=MODULE_CONFIG_JSON,
    population=MODEL_POPULATION,
    min_unique_exercises=None,
    min_activity_exercises=MIN_SEQUENCE_EXERCISES,
    keep_only_single_module_playlists=KEEP_ONLY_SINGLE_MODULE_PLAYLISTS,
    isolate_process=False,
    sequence_scope=SEQUENCE_SCOPE,
)
combined_trajectory = dataset.frame

population_attempt_summary = pd.DataFrame([dataset.audit])[
    [
        'population',
        'eligible_attempt_rows',
        'students',
        'classrooms',
        'modules',
        'exercises',
        'first_attempt_rows',
        'eligible_playlist_rows',
        'eligible_zpdes_rows',
    ]
]
display(population_attempt_summary)

Preparing combined population...


,population,eligible_attempt_rows,students,classrooms,modules,exercises,first_attempt_rows,eligible_playlist_rows,eligible_zpdes_rows
0,combined,5590740,37894,3091,27,18367,4666069,1633559,3957181


## 4. Keep first retained attempts and create module-sequence positions

Within the combined population, the earliest retained row for each student–exercise pair is kept. Later rows for that exercise are discarded even if they occur in another analysed work mode. The remaining exercises are ordered chronologically for each student, then split whenever the module or work mode changes. Changing activity alone does not restart position. Each retained run must contain more than 30 unique exercises.

For example, `zpdes(A), zpdes(B), zpdes(C)` forms one sequence across three activities. In contrast, `zpdes(A), playlist(B), zpdes(C)` forms three separate sequences because the work mode changes.

Important scope: because the shared loader has already removed adaptive-test, revision, and other modes, this is the **first retained playlist/zpdes attempt**. It is not guaranteed to be the first-ever encounter in the complete raw history.

In [40]:
trajectory_summary = pd.DataFrame([dataset.audit])[
    [
        'population', 'eligible_attempt_rows', 'first_attempt_rows',
        'trajectory_rows', 'removed_repeat_rows',
        'removed_short_segment_rows', 'students', 'classrooms',
        'modules', 'exercises', 'segments', 'max_attempt_position',
    ]
]
display(trajectory_summary)
display(combined_trajectory.head())

,population,eligible_attempt_rows,first_attempt_rows,trajectory_rows,removed_repeat_rows,removed_short_segment_rows,students,classrooms,modules,exercises,segments,max_attempt_position
0,combined,5590740,4666069,4067552,924671,598517,37894,3091,27,18367,43035,1044


,population,student_id,classroom_id,module,activity_id,module_sequence_id,work_mode,exercise_id,created_at,success,attempt_position,segment_exercises,student_in_classroom,is_first_retained_attempt
0,exclusive_modes,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,Améliorer la compréhension des textes,5d20f6e0-a60b-11ed-afa1-0242ac120002,1,zpdes,842d428d-489c-497a-b814-417c9a18b15f,2025-12-18 09:55:08.195000+00:00,1,0,100,132484cb-02b9-4cc2-974a-e38e485dd1f1 00017460-1206-45ee-b1d3-393c72a45220,True
1,exclusive_modes,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,Améliorer la compréhension des textes,5d20f6e0-a60b-11ed-afa1-0242ac120002,1,zpdes,4ddb6a28-e8c4-4cd1-94ce-5a44052b7dc3,2025-12-18 09:55:30.045000+00:00,1,1,100,132484cb-02b9-4cc2-974a-e38e485dd1f1 00017460-1206-45ee-b1d3-393c72a45220,True
2,exclusive_modes,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,Améliorer la compréhension des textes,5d20f6e0-a60b-11ed-afa1-0242ac120002,1,zpdes,8ed19610-59e8-48fc-9433-4755e6c31c71,2025-12-18 09:55:53.084000+00:00,1,2,100,132484cb-02b9-4cc2-974a-e38e485dd1f1 00017460-1206-45ee-b1d3-393c72a45220,True
3,exclusive_modes,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,Améliorer la compréhension des textes,73275ee6-2d21-482b-868e-73cd8cd34d91,1,zpdes,bd19af6e-592e-4cf2-ba20-06602956fc39,2025-12-18 09:56:22.733000+00:00,1,3,100,132484cb-02b9-4cc2-974a-e38e485dd1f1 00017460-1206-45ee-b1d3-393c72a45220,True
4,exclusive_modes,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,Améliorer la compréhension des textes,73275ee6-2d21-482b-868e-73cd8cd34d91,1,zpdes,0f81fedb-cd9d-4d86-ac3c-0a3c38b98239,2025-12-18 09:56:35.465000+00:00,1,4,100,132484cb-02b9-4cc2-974a-e38e485dd1f1 00017460-1206-45ee-b1d3-393c72a45220,True


## 5. Audit module-sequence length and position support

The tables below show module-sequence-length and attempt-position distributions. Counts decrease at later positions because only longer sequences contribute there. The figure will stop at the largest common position supported by at least `MIN_SEQUENCES_PER_POSITION` sequences and `MIN_STUDENTS_PER_POSITION` distinct students in each work mode.

In [41]:
segments = combined_trajectory.groupby(
    ['student_id', 'module', 'module_sequence_id', 'work_mode'],
    as_index=False,
    observed=True,
).agg(segment_exercises=('exercise_id', 'size'))
segment_distribution = (
    segments.groupby('work_mode', observed=True)['segment_exercises']
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .reset_index()
)
segment_distribution.insert(0, 'population', MODEL_POPULATION)

position_distribution = (
    combined_trajectory.groupby('work_mode', observed=True)['attempt_position']
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
    .reset_index()
)
position_distribution.insert(0, 'population', MODEL_POPULATION)
display(segment_distribution)
display(position_distribution)

,population,work_mode,count,mean,std,min,25%,50%,75%,90%,95%,max
0,combined,playlist,13102.0,89.071974,80.882894,31.0,44.0,64.0,104.0,164.0,219.0,1045.0
1,combined,zpdes,29933.0,96.900778,74.679561,31.0,47.0,72.0,119.0,191.0,250.0,719.0


,population,work_mode,count,mean,std,min,50%,75%,90%,95%,99%,max
0,combined,playlist,1167021.0,80.756534,104.874543,0.0,47.0,96.0,184.0,278.0,555.0,1044.0
1,combined,zpdes,2900531.0,76.726477,78.084222,0.0,51.0,103.0,178.0,236.0,367.0,718.0


## 6. Calculate observed success rate and mean exercise Elo

At a given attempt number, every eligible student/module/work-mode sequence contributes at most one binary first-attempt outcome. The observed success rate is the arithmetic mean of those 0/1 outcomes. It is a real empirical proportion, not a model prediction.

Exercise Elo is first joined using the exact `(exercise_id, activity_id)` context. Missing playlist activity UUIDs are filled from `config_mia.json` learning items; when an exact Elo context is still unavailable, the notebook falls back to the exercise ID. If that ID has several calibrated contexts, their Elo values are averaged. Only calibrated, non-missing Elo values contribute to the Elo mean; coverage and exact-versus-fallback match counts are reported. Higher Elo means a more difficult exercise.

In [42]:
exercise_elo = pd.read_parquet(
    EXERCISE_ELO_FILE,
    columns=['exercise_id', 'activity_id', 'exercise_elo', 'calibrated'],
)
empirical_trajectory = build_empirical_trajectory_summary(
    combined_trajectory,
    exercise_elo,
)

supported_positions = empirical_trajectory[
    (empirical_trajectory['sequences'] >= MIN_SEQUENCES_PER_POSITION)
    & (empirical_trajectory['students'] >= MIN_STUDENTS_PER_POSITION)
]
maximum_supported_position = (
    supported_positions.groupby('work_mode', observed=True)['attempt_number'].max()
)
missing_modes = {'playlist', 'zpdes'}.difference(maximum_supported_position.index.astype(str))
if missing_modes:
    raise ValueError(f'Insufficient positional support for: {sorted(missing_modes)}')
common_max_attempt = int(maximum_supported_position.min())
plot_trajectory = empirical_trajectory[
    empirical_trajectory['attempt_number'] <= common_max_attempt
].sort_values(['work_mode', 'attempt_number']).copy()
for metric in ('success_rate', 'mean_exercise_elo'):
    plot_trajectory[f'{metric}_smoothed'] = (
        plot_trajectory.groupby('work_mode', observed=True)[metric]
        .transform(
            lambda values: values.rolling(
                window=SMOOTHING_WINDOW,
                center=True,
                min_periods=1,
            ).mean()
        )
    )
display(empirical_trajectory.head(10))
print(
    f'Common displayed range: attempts 1 to {common_max_attempt} '
    f'(at least {MIN_SEQUENCES_PER_POSITION} sequences and '
    f'{MIN_STUDENTS_PER_POSITION} students per work mode)'
)

,work_mode,attempt_position,attempt_number,attempt_rows,sequences,students,modules,success_rate,mean_exercise_elo,elo_attempt_rows,exact_context_elo_attempt_rows,exercise_id_fallback_elo_attempt_rows,elo_coverage
0,playlist,0,1,13102,13102,8577,24,0.740956,1415.052938,13102,13102,0,1.0
1,playlist,1,2,13102,13102,8577,24,0.791711,1402.954274,13102,13102,0,1.0
2,playlist,2,3,13102,13102,8577,24,0.762937,1420.403150,13102,13102,0,1.0
3,playlist,3,4,13102,13102,8577,24,0.772020,1421.170357,13102,13102,0,1.0
4,playlist,4,5,13102,13102,8577,24,0.781636,1409.381993,13102,13102,0,1.0
5,playlist,5,6,13102,13102,8577,24,0.784537,1414.643731,13102,13102,0,1.0
6,playlist,6,7,13102,13102,8577,24,0.755228,1435.501520,13102,13102,0,1.0
7,playlist,7,8,13102,13102,8577,24,0.785834,1406.868080,13102,13102,0,1.0
8,playlist,8,9,13102,13102,8577,24,0.782934,1410.663445,13102,13102,0,1.0
9,playlist,9,10,13102,13102,8577,24,0.774233,1429.664773,13102,13102,0,1.0


Common displayed range: attempts 1 to 428 (at least 100 sequences and 100 students per work mode)


## 7. Read the descriptive values

For each work mode and attempt number:

- `success_rate` is the observed proportion of successful first attempts.
- `mean_exercise_elo` is the mean calibrated difficulty of the exercises presented.
- `sequences` is the number of eligible sequences contributing one observation.
- `students` and `modules` describe the support behind the point.
- `elo_coverage` is the proportion of attempts matched to a calibrated Elo value.
- `exact_context_elo_attempt_rows` and `exercise_id_fallback_elo_attempt_rows` distinguish exact context matches from exercise-ID fallbacks.

The exported values are not adjusted or extrapolated. A position is displayed only while both work modes meet the configured minimum numbers of contributing sequences and distinct students. For readability, the figure displays a small centered rolling average controlled by `SMOOTHING_WINDOW`; the exported table retains the unsmoothed observations.

In [43]:
elo_coverage_summary = (
    empirical_trajectory.groupby('work_mode', observed=True)
    .agg(
        attempt_rows=('attempt_rows', 'sum'),
        elo_attempt_rows=('elo_attempt_rows', 'sum'),
        exact_context_elo_attempt_rows=('exact_context_elo_attempt_rows', 'sum'),
        exercise_id_fallback_elo_attempt_rows=(
            'exercise_id_fallback_elo_attempt_rows', 'sum'
        ),
        minimum_position_coverage=('elo_coverage', 'min'),
    )
    .reset_index()
)
elo_coverage_summary['overall_elo_coverage'] = (
    elo_coverage_summary['elo_attempt_rows'] / elo_coverage_summary['attempt_rows']
)
elo_coverage_summary['fallback_share_of_matched_elo'] = (
    elo_coverage_summary['exercise_id_fallback_elo_attempt_rows']
    / elo_coverage_summary['elo_attempt_rows']
)
snapshot_positions = sorted({1, 10, 20, 31, common_max_attempt})
trajectory_snapshot = plot_trajectory[
    plot_trajectory['attempt_number'].isin(snapshot_positions)
][
    [
        'work_mode', 'attempt_number', 'success_rate', 'mean_exercise_elo',
        'sequences', 'students', 'modules', 'elo_coverage',
    ]
]
display(elo_coverage_summary.round(4))
display(trajectory_snapshot.round(4))

,work_mode,attempt_rows,elo_attempt_rows,exact_context_elo_attempt_rows,exercise_id_fallback_elo_attempt_rows,minimum_position_coverage,overall_elo_coverage,fallback_share_of_matched_elo
0,playlist,1167021,1167021,1167021,0,1.0,1.0,0.0
1,zpdes,2900531,2900531,2900531,0,1.0,1.0,0.0


,work_mode,attempt_number,success_rate,mean_exercise_elo,sequences,students,modules,elo_coverage
0,playlist,1,0.7410,1415.0529,13102,8577,24,1.0
9,playlist,10,0.7742,1429.6648,13102,8577,24,1.0
19,playlist,20,0.7368,1465.2386,13102,8577,24,1.0
30,playlist,31,0.7389,1449.0575,13102,8577,24,1.0
427,playlist,428,0.7820,1466.4827,133,109,12,1.0
1045,zpdes,1,0.7626,1377.0301,29933,19601,24,1.0
1054,zpdes,10,0.7156,1443.4771,29933,19601,24,1.0
1064,zpdes,20,0.7010,1491.5704,29933,19601,24,1.0
1075,zpdes,31,0.6844,1512.2069,29933,19601,24,1.0
1472,zpdes,428,0.6524,1578.6677,164,161,8,1.0


## 8. Plot observed success and selected exercise difficulty

Solid lines use the left axis and show observed first-attempt success rates. Dashed lines use the right axis and show mean calibrated exercise Elo; higher Elo means harder exercises. All displayed lines apply a light centered rolling average.

In [44]:
colors = {'playlist': '#DD8452', 'zpdes': '#4C72B0'}
figure = make_subplots(specs=[[{'secondary_y': True}]])
for work_mode in ('playlist', 'zpdes'):
    mode_data = plot_trajectory[plot_trajectory['work_mode'].eq(work_mode)]
    mode_label = work_mode.upper() if work_mode == 'zpdes' else work_mode.capitalize()
    figure.add_trace(
        go.Scatter(
            x=mode_data['attempt_number'],
            y=mode_data['success_rate_smoothed'],
            mode='lines',
            name=f'{mode_label} : taux de réussite',
            line={'color': colors[work_mode], 'width': 3},
            customdata=mode_data[['sequences', 'students', 'modules']].to_numpy(),
            hovertemplate=(
                'Attempt %{x}<br>Smoothed success rate: %{y:.1%}'
                '<br>Sequences: %{customdata[0]:,.0f}'
                '<br>Students: %{customdata[1]:,.0f}'
                '<br>Modules: %{customdata[2]:,.0f}<extra></extra>'
            ),
        ),
        secondary_y=False,
    )
    figure.add_trace(
        go.Scatter(
            x=mode_data['attempt_number'],
            y=mode_data['mean_exercise_elo_smoothed'],
            mode='lines',
            name=f'{mode_label} : difficulté des exercices (Elo)',
            line={'color': colors[work_mode], 'width': 2.5, 'dash': 'dash'},
            customdata=mode_data[['elo_coverage', 'elo_attempt_rows']].to_numpy(),
            hovertemplate=(
                'Attempt %{x}<br>Smoothed exercise Elo: %{y:.1f}'
                '<br>Elo coverage: %{customdata[0]:.1%}'
                '<br>Matched attempts: %{customdata[1]:,.0f}<extra></extra>'
            ),
        ),
        secondary_y=True,
    )

figure.update_layout(
    title='Observed first-attempt success and exercise difficulty',
    template='simple_white',
    xaxis_title='New-exercise number within module sequence',
    hovermode='x unified',
    font={'family': 'Arial', 'size': 13, 'color': '#333333'},
    title_x=0.5,
    title_y=0.99,
    title_yanchor='top',
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin={'l': 75, 'r': 75, 't': 115, 'b': 70},
    legend={
        'orientation': 'h',
        'y': 1.13,
        'x': 0.5,
        'xanchor': 'center',
    },
)
figure.update_xaxes(showgrid=False, zeroline=False, linecolor='#B8B8B8')
figure.update_yaxes(
    title_text='Observed first-attempt success rate',
    tickformat='.0%',
    range=[0, 1],
    showgrid=True,
    gridcolor='#E6E6E6',
    zeroline=False,
    secondary_y=False,
)
figure.update_yaxes(
    title_text='Mean calibrated exercise Elo (higher = harder)',
    showgrid=False,
    zeroline=False,
    secondary_y=True,
)
SVG_EXPORT_CONFIG = {
    'toImageButtonOptions': {
        'format': 'svg',
        'filename': 'observed_first_attempt_success_and_exercise_difficulty',
        'width': 1400,
        'height': 700,
        'scale': 1,
    },
    'displaylogo': False,
}
figure.show(config=SVG_EXPORT_CONFIG)

## 9. Interpretation limits

This is a descriptive observational trajectory, not a statistical model or causal estimate. Differences can reflect learning, student composition, selected content, exercise difficulty, or other features of the two work modes. Students with several eligible sequences can contribute to several points.

Exercise Elo is retrospective, module-local, and available only for calibrated exercise contexts. Playlist Elo often uses the exercise-ID fallback because playlist attempts do not carry the activity UUID; duplicated exercise IDs are represented by their mean calibrated Elo across contexts. The displayed range requires at least `MIN_SEQUENCES_PER_POSITION` sequences and `MIN_STUDENTS_PER_POSITION` distinct students in each mode, but the exported table includes the complete observed tail. Displayed curves use a centered rolling average over `SMOOTHING_WINDOW` positions. Smoothing affects only the display, and no extrapolation is applied.

## 10. Save outputs

In [45]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
population_attempt_summary.to_csv(OUTPUT_DIR / 'population_attempt_summary.csv', index=False)
trajectory_summary.to_csv(OUTPUT_DIR / 'first_attempt_trajectory_summary.csv', index=False)
segment_distribution.to_csv(OUTPUT_DIR / 'segment_length_distribution.csv', index=False)
position_distribution.to_csv(OUTPUT_DIR / 'attempt_position_distribution.csv', index=False)
empirical_trajectory.to_csv(
    OUTPUT_DIR / 'observed_success_and_exercise_elo_by_attempt.csv',
    index=False,
)
elo_coverage_summary.to_csv(OUTPUT_DIR / 'exercise_elo_coverage.csv', index=False)
trajectory_snapshot.to_csv(OUTPUT_DIR / 'observed_trajectory_snapshot.csv', index=False)
figure.write_html(
    OUTPUT_DIR / 'observed_success_and_exercise_elo_trajectory.html',
    include_plotlyjs='cdn',
    config=SVG_EXPORT_CONFIG,
)
print(f'Saved outputs to: {OUTPUT_DIR}')

Saved outputs to: c:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\artifacts\first_attempt_trajectory_mia_notebook
